# 01 · Explore the data

Run `python ml/generate_synthetic.py` and `python ml/download_datasets.py` first.
This notebook only looks at the data; training and scoring live in the `ml/*.py` scripts.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')
from features import build_features

room = pd.read_csv('../data/synthetic/train_room.csv.gz')
water = pd.read_csv('../data/synthetic/train_water.csv.gz')
cold = pd.read_csv('../data/synthetic/train_cold.csv.gz')
episodes = pd.read_csv('../data/synthetic/train_episodes.csv')
print(room.shape, water.shape, cold.shape)
episodes['fault'].value_counts()

In [ ]:
# One compressor failure: temperature climbs out of the safe band
e = episodes[episodes.fault == 'compressor_failure'].iloc[0]
d = cold[cold.device_id == e.device_id].copy()
d['t'] = pd.to_datetime(d.ts.str.slice(0, 19))
start = pd.to_datetime(e.start[:19])
w = d[(d.t > start - pd.Timedelta('60min')) & (d.t < start + pd.Timedelta('150min'))]
fig, ax = plt.subplots(figsize=(10, 3))
ax.axhspan(w.safe_min_c.iloc[0], w.safe_max_c.iloc[0], color='green', alpha=0.08, label='safe band')
ax.plot(w.t, w.temp_c, lw=2, label=e.device_id)
ax.axvline(start, color='grey', lw=1)
ax.set_ylabel('°C'); ax.legend(); ax.set_title('Compressor failure (fault starts at the grey line)')

In [ ]:
# The physics feature that separates a stuck AC from normal setback cooling
f = build_features(room.sort_values(['device_id', 'ts']).reset_index(drop=True), 'room')
f['label'] = room.sort_values(['device_id', 'ts']).reset_index(drop=True)['label']
f.groupby('label')[['below_setpoint_run', 'empty_cooling_30', 'temp_minus_setpoint']].describe().T

In [ ]:
# Real data: NAB machine temperature with its labelled failure windows
import json
nab = pd.read_csv('../data/raw/nab/machine_temperature_system_failure.csv', parse_dates=['timestamp'])
windows = json.load(open('../data/raw/nab/combined_windows.json'))['realKnownCause/machine_temperature_system_failure.csv']
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(nab.timestamp, nab.value, lw=1)
for a, b in windows:
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color='red', alpha=0.15)
ax.set_title('NAB machine temperature (red = labelled failure windows)')